# 01 · Data Collection
**MLS NLP Pipeline — Stage 1**

This notebook walks through how MLS press articles were collected for 2018–2024.

### Sources
- **GDELT Article Search API** — free historical news archive, queried by MLS-focused keywords
- **RSS Feeds** — MLS official, American Soccer Now, Soccer America, ESPN Soccer

### Design Principles
- Month-by-month collection with **SQLite checkpointing** (resume if interrupted)
- **SHA-256 content-hash deduplication** across all sources and runs
- Rate-limited GDELT requests (1.2 s between calls) to avoid throttling
- Full-text extraction via `trafilatura` (with `newspaper3k` fallback)

## 1. Project Setup

In [ ]:
import sys
from pathlib import Path

# Point to project root so pipeline imports resolve
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

from pipeline.utils import get_config, CheckpointDB, PROJECT_ROOT

settings = get_config("settings")
sources  = get_config("sources")

print("Data dir  :", PROJECT_ROOT / settings["pipeline"]["data_dir"])
print("State dir :", PROJECT_ROOT / settings["pipeline"]["state_dir"])
print("Years     :", sources["collection"]["start_year"], "–", sources["collection"]["end_year"])

## 2. GDELT Queries & RSS Feeds

In [ ]:
import pandas as pd

# What queries are configured?
gdelt_queries = sources.get("gdelt", {}).get("queries", [])
rss_feeds     = sources.get("rss",   {}).get("feeds",   [])

print(f"GDELT queries ({len(gdelt_queries)}):")
for q in gdelt_queries:
    print(f"  · {q['name']:<30}  max_records={q.get('max_records', 250)}")

print(f"\nRSS feeds ({len(rss_feeds)}):")
for f in rss_feeds:
    print(f"  · {f['name']}")

## 3. How the Collector Works

In [ ]:
import inspect
from pipeline.collector import GDELTCollector, TextExtractor

# Show the GDELT fetch method signature and key logic
print(inspect.getsource(GDELTCollector.fetch_month))

### Key Design: Timeout-Safe Full-Text Extraction

`trafilatura.fetch_url()` ignores Python timeouts. We fixed this by fetching with
`requests.get(url, timeout=8)` directly and passing the HTML to `trafilatura.extract()`:

In [ ]:
print(inspect.getsource(TextExtractor._extract_trafilatura))

## 4. Checkpoint Database

In [ ]:
db = CheckpointDB(str(PROJECT_ROOT / "state" / "pipeline_state.db"))

# How many months are marked complete for the collection stage?
import sqlite3
con = sqlite3.connect(str(PROJECT_ROOT / "state" / "pipeline_state.db"))
cur = con.cursor()

cur.execute("SELECT status, COUNT(*) FROM checkpoints WHERE stage='collector' GROUP BY status")
rows = cur.fetchall()
print("Collection checkpoints:")
for status, cnt in rows:
    print(f"  {status:<12} {cnt} months")

cur.execute("SELECT COUNT(*) FROM seen_articles")
print(f"\nTotal deduplicated articles tracked: {cur.fetchone()[0]:,}")
con.close()

## 5. Collected Data — Sample

In [ ]:
import pandas as pd
from pipeline.utils import load_parquet, get_parquet_path

data_dir = PROJECT_ROOT / settings["pipeline"]["data_dir"]

# Load a sample month
path = get_parquet_path(data_dir / "press" / "raw", "raw", 2023, 7)
df   = load_parquet(path)

print(f"2023-07: {len(df)} articles")
df[["title", "domain", "published_date", "has_full_text", "text_length"]].head(10)

## 6. Collection Volume Over Time

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

records = []
for year in range(2018, 2025):
    for month in range(1, 13):
        path = get_parquet_path(data_dir / "press" / "raw", "raw", year, month)
        df_m = load_parquet(path)
        if not df_m.empty:
            records.append({"year": year, "month": month,
                            "articles": len(df_m),
                            "with_text": df_m["has_full_text"].sum()})

vol = pd.DataFrame(records)
vol["ym"] = vol["year"].astype(str) + "-" + vol["month"].apply(lambda m: f"{m:02d}")

fig, ax = plt.subplots(figsize=(15, 4))
ax.bar(range(len(vol)), vol["articles"],  label="Total articles",   color="#3498db", alpha=0.7)
ax.bar(range(len(vol)), vol["with_text"], label="Full text extracted", color="#27ae60", alpha=0.9)

# Year dividers
for i, row in vol[vol["month"] == 1].iterrows():
    ax.axvline(i, color="gray", lw=0.8, ls="--", alpha=0.5)
    ax.text(i + 0.3, ax.get_ylim()[1] * 0.95, str(row["year"]), fontsize=8, color="gray")

ax.set_xticks(range(0, len(vol), 3))
ax.set_xticklabels(vol["ym"].iloc[::3], rotation=45, ha="right", fontsize=7)
ax.set_ylabel("Article count")
ax.set_title("Monthly Article Collection Volume (2018–2024)")
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nTotal articles collected: {vol['articles'].sum():,}")
print(f"Full text extracted:       {vol['with_text'].sum():,} ({vol['with_text'].sum()/vol['articles'].sum()*100:.1f}%)")

## 7. Domain Distribution

In [ ]:
from pipeline.utils import load_all_parquet

all_raw = load_all_parquet(data_dir / "press" / "raw")
top_domains = (all_raw["domain"]
               .value_counts()
               .head(20)
               .reset_index()
               .rename(columns={"index": "domain", "domain": "count"}))
print(top_domains.to_string(index=False))